# TITAN similarity search example

This notebook demonstrates a simple retrieval workflow: (1) download all released TITAN slide embeddings, (2) find the most similar patient cases to a query slide using cosine similarity, (3) retrieve public clinical annotations from CCDI cBioPortal, and (4) print IDC viewer links and raw-image download command for further inspection.

The clinical annotations are shown only to describe the retrieved cases, they are not used to calculate similarity. The embeddings and retrieval results are intended for research use and have not been validated for clinical diagnosis. 

## Setup

`huggingface_hub`, `requests`, `pandas`, `numpy`, and `h5py` are needed.

In [ ]:
%pip install -q huggingface_hub requests pandas numpy h5py

In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import requests
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path("ccdi_mci_demo_data")
QUERY_PATIENT_ID = "PANLMU"
QUERY_SERIES_UID = None  # Set this when a patient has multiple slides.
TOP_K = 5

REPO_ID = "CBIIT-CGBB/CCDI-MCI"
CBIO_STUDY_ID = "phs002790"
CBIO_API = "https://cbioportal-api.ccdi.cancer.gov/api"

/data/nextgen/keshk/GBM_MRI/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download the manifest and all TITAN embeddings

+ The complete TITAN embedding set is approximately 660 MB.
+ The broad pattern avoids constructing thousands of patient-specific patterns.
+ Re-running the cell reuses files already present in `DATA_DIR`.

In [ ]:
manifest_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="metadata/slide_manifest.csv",
    local_dir=DATA_DIR,
)

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns="embeddings/titan/*/*.h5",
    local_dir=DATA_DIR,
    max_workers=16,
)

manifest = pd.read_csv(manifest_path)
embedding_root = DATA_DIR / "embeddings" / "titan"
search_table = manifest.loc[manifest["has_titan"].fillna(False)].copy()
search_table["feature_path"] = search_table.apply(
    lambda row: embedding_root / row["patient_id"] / f"{row['series_instance_uid']}.h5",
    axis=1,
)

missing = search_table.loc[~search_table["feature_path"].map(Path.is_file)]

if not missing.empty:
    raise FileNotFoundError(f"Missing {len(missing)} TITAN files after download")

search_table = search_table.reset_index(drop=True)
print(f"Ready to search through {len(search_table)} TITAN slide embeddings")

Fetching 4570 files: 100%|██████████| 4570/4570 [00:10<00:00, 417.63it/s]


Ready to search 4570 TITAN slide embeddings


## Load and normalize the 768-dimensional slide embeddings

All vectors are small enough to keep in memory. L2 normalization makes their dot product equal to cosine similarity.

In [ ]:
def load_titan_embedding(path):
    with h5py.File(path, "r") as handle:
        embedding = handle["features"][:].astype(np.float32).squeeze()
    return embedding


embeddings = np.stack(search_table["feature_path"].map(load_titan_embedding))
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized_embeddings = embeddings / norms
print(f"Embedding matrix: {normalized_embeddings.shape}")

Embedding matrix: (4570, 768)


## Find the top-k most similar patient cases

The query is selected by patient ID and, optionally, a SeriesInstanceUID. Every slide is scored, then the highest-scoring slide from each other patient is retained

In [ ]:
query_candidates = search_table.loc[search_table["patient_id"].eq(QUERY_PATIENT_ID)]

if QUERY_SERIES_UID is not None:
    query_candidates = query_candidates.loc[
        query_candidates["series_instance_uid"].eq(QUERY_SERIES_UID)
    ]
if query_candidates.empty:
    raise ValueError("The requested query patient/series is not in the TITAN release")

query_index = query_candidates.index[0]
query = search_table.loc[query_index]

if QUERY_SERIES_UID is None and len(query_candidates) > 1:
    print(f"{QUERY_PATIENT_ID} has {len(query_candidates)} slides; using the first one in the manifest")

search_table["cosine_similarity"] = normalized_embeddings @ normalized_embeddings[query_index]
results = (
    search_table.loc[~search_table["patient_id"].eq(query["patient_id"])]
    .sort_values("cosine_similarity", ascending=False)
    .drop_duplicates("patient_id")
    .head(TOP_K)
    .copy()
)
results.insert(0, "rank", np.arange(1, len(results) + 1))

print(f"Query patient: {query['patient_id']}")
print(f"Query series:  {query['series_instance_uid']}")

Query patient: PANLMU
Query series:  1.3.6.1.4.1.5962.99.1.856942911.401081431.1727433828671.4.0


## Add public clinical annotations from CCDI cBioPortal

`MCI Primary Disease Group` and `Methylation Classification` are patient-level attributes obtained from cBioPortal

In [5]:
def fetch_clinical_data(clinical_data_type):
    response = requests.get(
        f"{CBIO_API}/studies/{CBIO_STUDY_ID}/clinical-data",
        params={"clinicalDataType": clinical_data_type},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def collect_attribute(records, attribute_id, column_name):
    values = pd.DataFrame(
        {"patient_id": record["patientId"], column_name: record["value"]}
        for record in records
        if record.get("clinicalAttributeId") == attribute_id and record.get("value")
    )
    if values.empty:
        return pd.DataFrame(columns=["patient_id", column_name])
    return (
        values.groupby("patient_id", as_index=False)[column_name]
        .agg(lambda items: " | ".join(sorted(set(items))))
    )


sample_records = fetch_clinical_data("SAMPLE")
patient_records = fetch_clinical_data("PATIENT")

clinical = (
    collect_attribute(patient_records, "PRIMDXDSCAT", "MCI Primary Disease Group")
    .merge(
        collect_attribute(sample_records, "CANCER_TYPE", "Cancer Type"),
        on="patient_id",
        how="outer",
    )
    .merge(
        collect_attribute(patient_records, "METH_CLASS", "Methylation Classification"),
        on="patient_id",
        how="outer",
    )
)

result_columns = [
    "rank",
    "cosine_similarity",
    "patient_id",
    "series_instance_uid",
    "MCI Primary Disease Group",
    "Cancer Type",
    "Methylation Classification",
    "idc_viewer_url",
]
results = results.merge(clinical, on="patient_id", how="left")[result_columns]
display(results)

,rank,cosine_similarity,patient_id,series_instance_uid,MCI Primary Disease Group,Cancer Type,Methylation Classification,idc_viewer_url
0,1,0.914618,PBCZLV,1.3.6.1.4.1.5962.99.1.3294175225.48184472.1755...,Central Nervous System,Other CNS,"Consistent with Meningioma, benign",https://viewer.imaging.datacommons.cancer.gov/...
1,2,0.865908,PBCDNA,1.3.6.1.4.1.5962.99.1.806797136.1785492695.172...,Central Nervous System,Other CNS,"Consistent with Meningioma, subclass benign 1",https://viewer.imaging.datacommons.cancer.gov/...
2,3,0.843233,PBCCTE,1.3.6.1.4.1.5962.99.1.925964868.1666902534.172...,Central Nervous System,Other CNS,"Consistent with Meningioma, subclass intermedi...",https://viewer.imaging.datacommons.cancer.gov/...
3,4,0.841770,PBCKPU,1.3.6.1.4.1.5962.99.1.477325463.512356295.1739...,Central Nervous System,Other CNS,NaN,https://viewer.imaging.datacommons.cancer.gov/...
4,5,0.827716,PBCWER,1.3.6.1.4.1.5962.99.1.3121606530.1703879248.17...,Central Nervous System,Other CNS,"Consistent with Meningioma, benign",https://viewer.imaging.datacommons.cancer.gov/...


## IDC links and raw-image download commands

The SeriesInstanceUID command downloads the exact slide represented by the result. The patient ID command downloads all IDC imaging series associated with that patient. These commands are printed only for reference. Install the current [`idc-index`](https://github.com/ImagingDataCommons/idc-index) package before running a command.

In [6]:
for row in results.itertuples(index=False):
    print(f"#{row.rank} | {row.patient_id} | cosine similarity: {row.cosine_similarity:.4f}")
    print(f"SLIM viewer: {row.idc_viewer_url}")
    print(f"Exact slide: idc download {row.series_instance_uid}")
    print(f"All patient images: idc download {row.patient_id}")
    print()

#1 | PBCZLV | cosine similarity: 0.9146
SLIM viewer: https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.154517207997491091983560266634519837837/series/1.3.6.1.4.1.5962.99.1.3294175225.48184472.1755640864761.4.0
Exact slide: idc download 1.3.6.1.4.1.5962.99.1.3294175225.48184472.1755640864761.4.0
All patient images: idc download PBCZLV

#2 | PBCDNA | cosine similarity: 0.8659
SLIM viewer: https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.279496800360962512059139530259358055608/series/1.3.6.1.4.1.5962.99.1.806797136.1785492695.1727383682896.4.0
Exact slide: idc download 1.3.6.1.4.1.5962.99.1.806797136.1785492695.1727383682896.4.0
All patient images: idc download PBCDNA

#3 | PBCCTE | cosine similarity: 0.8432
SLIM viewer: https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.206959021072806805062542799939359507855/series/1.3.6.1.4.1.5962.99.1.925964868.1666902534.1727502850628.4.0
Exact slide: idc download 1.3.6.1.4.1.5962.99.1.925964868.1666902534.17

## Potential additions

+ Set `QUERY_SERIES_UID` to compare a specific slide for a patient with multiple slides.
+ Retain multiple slides per patient instead of one representative result.
+ Add a small search interface or visualize the retrieved slides in a gallery.
+ Compare cosine retrieval across TITAN, CONCH v1.5, and UNI2-h representations.